In [ ]:
vm_query_url = ""
vm_token = ""
vm_jobs_notebook_env = ""
vm_jobs_config_path = ""
extractor_job_ids = []
expected_metric_name = "expected_number_of_metrics_count"
business_date = ""
lookback_days = 14
output_results_path = ""

In [ ]:
import json
import pandas as pd

# Show full dataframes in notebook output
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

# Match plotting approach used by metrics_forecast_notebooks notebooks
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (15, 8)

from report_helpers import compute_report_rows, get_latest_exported_biz_dates

In [ ]:
rows = compute_report_rows(
    vm_query_url=vm_query_url,
    vm_token=vm_token,
    extractor_job_ids=extractor_job_ids,
    expected_metric_name=expected_metric_name,
    business_date=business_date,
    lookback_days=int(lookback_days),
)

latest_exported_dates_raw = {}
latest_exported_dates = {}
latest_exported_dates_display = {}
try:
    latest_exported_dates_raw = get_latest_exported_biz_dates(
        extractor_job_ids=extractor_job_ids,
        config_path=vm_jobs_config_path,
        environment=vm_jobs_notebook_env,
    )
    latest_exported_dates = {
        job_id: pd.to_datetime(biz_date).normalize()
        for job_id, biz_date in latest_exported_dates_raw.items()
    }
    latest_exported_dates_display = {
        job_id: dt.strftime("%d/%m/%Y")
        for job_id, dt in latest_exported_dates.items()
    }
except Exception as exc:
    print(f"Could not load latest exported biz_date markers from DB: {exc}")

df = pd.DataFrame(rows)
if df.empty:
    print("No data returned for selected range")
    df_expected = pd.DataFrame()
    df_expected_chart = pd.DataFrame()
    wrong_format_df = pd.DataFrame()
else:
    # Ensure expected columns exist for downstream grouping/output.
    defaults = {
        "biz_date_raw": "",
        "biz_date_format": "invalid",
        "biz_date_normalized": "",
        "is_wrong_format": False,
    }
    for col, default in defaults.items():
        if col not in df.columns:
            df[col] = default

    # Build normalized date for all parseable formats so df_expected also shows wrong-format rows.
    df_expected = df[df["biz_date_format"].isin(["ddmm", "ddmm_ambiguous", "mmdd"])].copy()
    df_expected["biz_date"] = pd.to_datetime(
        df_expected["biz_date_normalized"], errors="coerce"
    ).dt.normalize()
    df_expected = df_expected[df_expected["biz_date"].notna()].copy()
    df_expected["latest_exported_biz_date"] = df_expected["job_id"].map(
        latest_exported_dates_display
    ).fillna("")

    # Keep existing chart behavior: expected-format only.
    df_expected_chart = df_expected[
        df_expected["biz_date_format"].isin(["ddmm", "ddmm_ambiguous"])
    ].copy()

    # Wrong format visibility: valid mm/dd rows only.
    wrong_format_df = df_expected[df_expected["biz_date_format"] == "mmdd"].copy()
    wrong_format_df["normalized_date"] = wrong_format_df["biz_date"]

    # Compare normalized mm/dd dates to dates present in expected-format data per job.
    expected_dates_by_job = (
        df_expected_chart.groupby("job_id")["biz_date"].apply(lambda s: set(s.dropna())).to_dict()
        if not df_expected_chart.empty
        else {}
    )
    wrong_format_df["exists_in_expected_dates"] = wrong_format_df.apply(
        lambda row: row["normalized_date"] in expected_dates_by_job.get(row["job_id"], set()),
        axis=1,
    )

    display(df_expected)

In [ ]:
import io
import matplotlib.dates as mdates
from IPython.display import Image, display

if not df_expected_chart.empty:
    for job_id in sorted(df_expected_chart["job_id"].unique()):
        job_df = df_expected_chart[df_expected_chart["job_id"] == job_id].copy()
        if job_df.empty:
            continue

        fig, ax = plt.subplots(figsize=(18, 8))

        expected_by_date = job_df.groupby("biz_date")["expected_count"].max().sort_index()
        actual_by_date = job_df.groupby("biz_date")["actual_count"].sum().sort_index()

        expected_color = "#1f4e79"
        actual_color = "#2e8b57"
        marker_color = "#6b7280"
        latest_color = "#d90429"

        ax.plot(
            expected_by_date.index,
            expected_by_date.values,
            label="Expected",
            color=expected_color,
            marker="o",
            linewidth=2.6,
            markersize=5,
            zorder=3,
        )
        ax.plot(
            actual_by_date.index,
            actual_by_date.values,
            label="Actual (all attempts)",
            color=actual_color,
            marker="o",
            linewidth=2.6,
            markersize=5,
            zorder=3,
        )

        for _, row in job_df.sort_values(["biz_date", "attempt"]).iterrows():
            ax.scatter(
                row["biz_date"],
                row["actual_count"],
                s=26,
                alpha=0.45,
                color=marker_color,
                edgecolor="white",
                linewidth=0.5,
                zorder=4,
            )

        latest_exported_date = latest_exported_dates.get(job_id)
        latest_exported_date_label = (
            latest_exported_date.strftime("%d/%m/%Y") if latest_exported_date is not None else ""
        )
        if latest_exported_date is not None:
            ax.axvline(
                latest_exported_date,
                color=latest_color,
                linestyle="--",
                linewidth=2.2,
                alpha=0.95,
                label=f"Latest exported biz_date: {latest_exported_date_label}",
                zorder=2,
            )
            ymax = max(
                float(expected_by_date.max()) if not expected_by_date.empty else 0.0,
                float(actual_by_date.max()) if not actual_by_date.empty else 0.0,
            )
            y_text = ymax * 0.94 if ymax > 0 else 1.0

            right_side = latest_exported_date >= expected_by_date.index.max() - pd.Timedelta(days=2)
            x_offset = -8 if right_side else 8
            h_align = "right" if right_side else "left"

            ax.annotate(
                f"Latest exported\n{latest_exported_date_label}",
                xy=(latest_exported_date, y_text),
                xytext=(x_offset, 0),
                textcoords="offset points",
                color=latest_color,
                fontsize=9,
                fontweight="bold",
                va="top",
                ha=h_align,
                bbox={"boxstyle": "round,pad=0.28", "fc": "white", "ec": latest_color, "alpha": 0.9},
            )

        ax.set_title(
            f"{job_id}: Expected vs Actual Metrics by Business Date",
            fontsize=14,
            fontweight="bold",
            pad=14,
        )
        ax.set_xlabel("Business Date", fontsize=11)
        ax.set_ylabel("Metrics Count", fontsize=11)

        ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=6, maxticks=12))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m/%Y"))
        for label in ax.get_xticklabels():
            label.set_rotation(25)
            label.set_ha("right")

        y_max = max(
            float(expected_by_date.max()) if not expected_by_date.empty else 0.0,
            float(actual_by_date.max()) if not actual_by_date.empty else 0.0,
        )
        ax.set_ylim(bottom=0, top=(y_max * 1.15 if y_max > 0 else 1.0))

        ax.grid(True, alpha=0.2, linestyle="-")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.legend(loc="upper left", frameon=True, facecolor="white", framealpha=0.9)
        fig.tight_layout()

        # Explicitly embed the figure output (reliable in papermill/headless -> nbconvert HTML)
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=180, bbox_inches="tight")
        buf.seek(0)
        display(Image(data=buf.getvalue()))
        plt.close(fig)
else:
    print("No expected-format (dd/mm) rows available for charting")

In [ ]:
if wrong_format_df.empty:
    print("No wrong-format biz_date rows found")
else:
    wrong_view = wrong_format_df[
        [
            "job_id",
            "biz_date_raw",
            "biz_date_normalized",
            "normalized_date",
            "exists_in_expected_dates",
            "attempt",
            "actual_count",
            "expected_count",
        ]
    ].copy()
    wrong_view["biz_date_normalized"] = pd.to_datetime(
        wrong_view["biz_date_normalized"], errors="coerce"
    ).dt.strftime("%d/%m/%Y")
    wrong_view["normalized_date"] = pd.to_datetime(
        wrong_view["normalized_date"], errors="coerce"
    ).dt.strftime("%d/%m/%Y")

    wrong_view = wrong_view.rename(
        columns={
            "biz_date_raw": "biz_date_label_raw",
            "biz_date_normalized": "normalized_ddmmyyyy",
            "normalized_date": "normalized_date_ddmmyyyy",
        }
    )
    wrong_view = wrong_view.sort_values(
        ["job_id", "normalized_ddmmyyyy", "attempt"]
    ).reset_index(drop=True)

    print("Wrong-format rows (dates shown as dd/mm/yyyy, grouped per job):")
    display(wrong_view)

In [ ]:
if output_results_path:
    payload = {
        "reports_generated": int(df_expected_chart["job_id"].nunique()) if not df_expected_chart.empty else 0,
        "rows": int(len(df)),
        "rows_expected_format": int(len(df_expected_chart)),
        "rows_wrong_format_mmdd": int(len(wrong_format_df)),
        "rows_ambiguous_ddmm": int((df["biz_date_format"] == "ddmm_ambiguous").sum()) if not df.empty else 0,
        "rows_invalid_biz_date": int((df["biz_date_format"] == "invalid").sum()) if not df.empty else 0,
        "wrong_format_distinct_dates": int(wrong_format_df["biz_date_normalized"].nunique()) if not wrong_format_df.empty else 0,
        "wrong_format_dates_existing_in_expected": int(
            wrong_format_df[wrong_format_df["exists_in_expected_dates"]]["biz_date_normalized"].nunique()
        ) if not wrong_format_df.empty else 0,
    }
    with open(output_results_path, "w", encoding="utf-8") as f:
        json.dump(payload, f)
    print(f"Saved run summary to {output_results_path}")